# Benchmarking Intent Classification: DoRA vs. LoRA

This notebook performs a head-to-head comparison between our **Finetuned DoRA model** and an **existing LoRA repository** model. We evaluate their ability to accurately detect customer intents within the **Banking77** dataset.

### Evaluation Workflow:
1.  **Inference Execution:** Run both models on the raw test set using consistent chat templates.
2.  **Performance Metrics:** Compute and compare **Accuracy, Macro F1, Precision, and Recall** to determine which model handles the 77 intent classes more effectively.
3.  **Error Profiling:** Generate **Confusion Matrices** and **Error Logs** to identify specific intents where models struggle or show significant improvement.
4.  **Benchmarking:** Establish a leaderboard to visualize the performance delta between the two PEFT (Parameter-Efficient Fine-Tuning) approaches.

---
**Models Comparison:**
*   **Our Model (DoRA):** `tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification`
*   **Existing Repo (LoRA):** `rajo0113/banking77-llama-1b-lora`

**Dataset:**
*   **Path:** `/kaggle/input/datasets/tranphungdinh/banking77-hf-split/split` (Test Split)

## 1. Install Dependencies

In [1]:
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade transformers trl liger-kernel datasets
!pip install -q scikit-learn seaborn matplotlib pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.4 MB

## 2. Imports & Core Functions 

In [2]:
import os
import json
import torch
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk
from tqdm.auto import tqdm
from unsloth import FastLanguageModel
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, confusion_matrix
import warnings
import logging
from transformers import logging as transformers_logging

# Create output directories
os.makedirs("outputs/metrics", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)
os.makedirs("outputs/error_analysis", exist_ok=True)


warnings.filterwarnings("ignore")
transformers_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)

# 1. METRICS COMPUTATION FUNCTION
def compute_metrics(true_labels, predicted_labels, model_name):
    metrics = {
        "exact_match_accuracy": accuracy_score(true_labels, predicted_labels),
        "macro_f1": f1_score(true_labels, predicted_labels, average="macro", zero_division=0),
        "macro_precision": precision_score(true_labels, predicted_labels, average="macro", zero_division=0),
        "macro_recall": recall_score(true_labels, predicted_labels, average="macro", zero_division=0),
    }
    
    print(f"\n[{model_name.upper()}] EVALUATION RESULTS:")
    print(f"Accuracy:  {metrics['exact_match_accuracy']:.4f}")
    print(f"F1 Score:  {metrics['macro_f1']:.4f}")
    print(f"Precision: {metrics['macro_precision']:.4f}")
    print(f"Recall:    {metrics['macro_recall']:.4f}")

    with open(f"outputs/metrics/{model_name}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    report = classification_report(true_labels, predicted_labels, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(f"outputs/metrics/{model_name}_per_class_report.csv")
    return metrics

# 2. CONFUSION MATRIX GENERATION FUNCTION
def generate_confusion_matrix(true_labels, predicted_labels, model_name):
    label_names = sorted(list(set(true_labels) | set(predicted_labels)))
    cm = confusion_matrix(true_labels, predicted_labels, labels=label_names)
    cm_normalized = cm.astype("float") / (cm.sum(axis=1, keepdims=True) + 1e-10)

    fig, ax = plt.subplots(figsize=(28, 24))
    sns.heatmap(
        cm_normalized, annot=False, cmap="YlOrRd",
        xticklabels=label_names, yticklabels=label_names,
        ax=ax, vmin=0, vmax=1, cbar_kws={"label": "Recall Proportion"}
    )
    ax.set_xlabel("Predicted Intent", fontsize=12)
    ax.set_ylabel("True Intent", fontsize=12)
    ax.set_title(f"Banking77 - {model_name} Confusion Matrix", fontsize=16)
    
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(rotation=0, fontsize=6)
    plt.tight_layout()
    fig.savefig(f"outputs/plots/{model_name}_confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

# 3. ERROR ANALYSIS FUNCTION
def analyze_errors(true_labels, predicted_labels, texts, model_name):
    error_pairs = []
    error_records = []
    for text, true, pred in zip(texts, true_labels, predicted_labels):
        if true != pred:
            error_pairs.append((true, pred))
            error_records.append({"text": text, "true_intent": true, "predicted_intent": pred})

    from collections import Counter
    top_pairs = Counter(error_pairs).most_common(20)
    confused_df = pd.DataFrame([
        {"true_intent": p[0], "predicted_as": p[1], "count": c} for p, c in top_pairs
    ])
    confused_df.to_csv(f"outputs/error_analysis/{model_name}_top_confused.csv", index=False)
    pd.DataFrame(error_records).to_csv(f"outputs/error_analysis/{model_name}_full_errors.csv", index=False)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. Load Dataset & Inference 

In [3]:
print("1. Loading dataset from your local path...")
DATASET_PATH = "/kaggle/input/datasets/tranphungdinh/banking77-hf-split/split"
dataset = load_from_disk(DATASET_PATH)

test_ds = dataset["test"]
texts = test_ds["text"]
true_intents = test_ds["label"]

print(f"✅ Successfully loaded {len(texts)} test samples!")

def run_inference(model_id, model_name_to_save):
    print(f"\n========== LOADING & PREPARING MODEL: {model_name_to_save} ==========")
    print(f"HF ID: {model_id}")
    
    # Unsloth automatically loads the base model + LoRA adapters from HF and merges them
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=512,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)  # Optimize VRAM for inference
    
    predictions = []
    
    print(f"Starting inference on {len(texts)} samples...")
    for text in tqdm(texts):
        # Build prompt to force the model to output only the intent name
        messages = [
            {"role": "system", "content": "You are an expert banking intent classifier. Given a user query, output ONLY the exact intent name from the Banking77 dataset. Do not explain."},
            {"role": "user", "content": text}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        
        # Generate prediction
        outputs = model.generate(
            **inputs, 
            max_new_tokens=15, 
            use_cache=True, 
            pad_token_id=tokenizer.eos_token_id
        )
        
        # Extract the generated output
        output_text = tokenizer.decode(
            outputs[0][inputs.input_ids.shape[1]:], 
            skip_special_tokens=True
        ).strip()
        
        predictions.append(output_text)
        
    # Clear VRAM to free space for the next model
    del model
    del tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    
    return predictions

1. Loading dataset from your local path...
✅ Successfully loaded 3080 test samples!


## 4. Process Finetuned LoRA Model

In [4]:
LORA_MODEL_ID = "rajo0113/banking77-llama-1b-lora"

pred_lora = run_inference(LORA_MODEL_ID, "LoRA_Model")

compute_metrics(true_intents, pred_lora, "LoRA_Model")
generate_confusion_matrix(true_intents, pred_lora, "LoRA_Model")
analyze_errors(true_intents, pred_lora, texts, "LoRA_Model")


========== LOADING & PREPARING MODEL: LoRA_Model ==========
HF ID: rajo0113/banking77-llama-1b-lora
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

Starting inference on 3080 samples...


  0%|          | 0/3080 [00:00<?, ?it/s]


[LORA_MODEL] EVALUATION RESULTS:
Accuracy:  0.8744
F1 Score:  0.7493
Precision: 0.7567
Recall:    0.7481


## 5. Process Finetuned DoRA Model

In [5]:
# Run evaluation on your fine-tuned DoRA model
DORA_MODEL_ID = "tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification"

pred_dora = run_inference(DORA_MODEL_ID, "DoRA_Model")

compute_metrics(true_intents, pred_dora, "DoRA_Model")
generate_confusion_matrix(true_intents, pred_dora, "DoRA_Model")
analyze_errors(true_intents, pred_dora, texts, "DoRA_Model")


========== LOADING & PREPARING MODEL: DoRA_Model ==========
HF ID: tpdinh2612/Llama-3.2-1B-Banking77-Intent-Classification
==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.8.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Starting inference on 3080 samples...


  0%|          | 0/3080 [00:00<?, ?it/s]


[DORA_MODEL] EVALUATION RESULTS:
Accuracy:  0.9231
F1 Score:  0.8377
Precision: 0.8420
Recall:    0.8362


## 6. Comparision

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# 1. Load metrics files
with open("outputs/metrics/LoRA_Model_metrics.json") as f:
    lora_metrics = json.load(f)
with open("outputs/metrics/DoRA_Model_metrics.json") as f:
    dora_metrics = json.load(f)

# 2. Create comparison DataFrame
comparison_df = pd.DataFrame(
    [lora_metrics, dora_metrics], 
    index=["LoRA Model (rajo0113)", "DoRA Model (tpdinh2612)"]
).T

# 3. Compute Delta and % Improvement (DoRA - LoRA)
comparison_df["Δ (DoRA - LoRA)"] = comparison_df["DoRA Model (tpdinh2612)"] - comparison_df["LoRA Model (rajo0113)"]
comparison_df["Improvement (%)"] = (comparison_df["Δ (DoRA - LoRA)"] / (comparison_df["LoRA Model (rajo0113)"] + 1e-10)) * 100

# 4. Format table for better readability
formatted_df = comparison_df.copy()

# Format original metrics to 4 decimal places
formatted_df["LoRA Model (rajo0113)"] = formatted_df["LoRA Model (rajo0113)"].apply(lambda x: f"{x:.4f}")
formatted_df["DoRA Model (tpdinh2612)"] = formatted_df["DoRA Model (tpdinh2612)"].apply(lambda x: f"{x:.4f}")

# Format Delta with +/- sign
formatted_df["Δ (DoRA - LoRA)"] = formatted_df["Δ (DoRA - LoRA)"].apply(lambda x: f"{'+' if x > 0 else ''}{x:.4f}")

# Format Improvement with % and +/- sign
formatted_df["Improvement (%)"] = formatted_df["Improvement (%)"].apply(lambda x: f"{'+' if x > 0 else ''}{x:.2f}%")

# 5. Display results with styling
print("\n" + "="*100)
print("🏆 PERFORMANCE COMPARISON: DoRA vs LoRA (Banking77 Intent Classification) 🏆")
print("="*100 + "\n")

# If running in Kaggle/Jupyter Notebook, display() renders a nice HTML table
try:
    display(formatted_df)
except NameError:
    # Fallback to markdown/text
    print(formatted_df.to_markdown())

# 6. DETAILED IMPROVEMENT ANALYSIS
print("\n" + "="*100)
print("📊 DETAILED IMPROVEMENT ANALYSIS")
print("="*100 + "\n")

improvement_summary = {
    "Metric": [],
    "LoRA": [],
    "DoRA": [],
    "Absolute Δ": [],
    "Relative Δ (%)": [],
    "Better": []
}

for metric in comparison_df.index:
    lora_val = lora_metrics[metric]
    dora_val = dora_metrics[metric]
    delta = dora_val - lora_val
    rel_delta = (delta / (lora_val + 1e-10)) * 100
    
    improvement_summary["Metric"].append(metric.replace("_", " ").title())
    improvement_summary["LoRA"].append(f"{lora_val:.4f}")
    improvement_summary["DoRA"].append(f"{dora_val:.4f}")
    improvement_summary["Absolute Δ"].append(f"{'+' if delta > 0 else ''}{delta:.4f}")
    improvement_summary["Relative Δ (%)"].append(f"{'+' if rel_delta > 0 else ''}{rel_delta:.2f}%")
    improvement_summary["Better"].append("🎯 DoRA" if delta > 0 else "❌ LoRA")

summary_df = pd.DataFrame(improvement_summary)
try:
    display(summary_df)
except NameError:
    print(summary_df.to_markdown(index=False))

# 7. VISUALIZATION: Improvement Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))

metrics_names = list(comparison_df.index)
lora_vals = [lora_metrics[m] for m in metrics_names]
dora_vals = [dora_metrics[m] for m in metrics_names]

x = range(len(metrics_names))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], lora_vals, width, label="LoRA Model", alpha=0.8, color="#FF6B6B")
bars2 = ax.bar([i + width/2 for i in x], dora_vals, width, label="DoRA Model", alpha=0.8, color="#4ECDC4")

ax.set_xlabel("Metrics", fontsize=12, fontweight="bold")
ax.set_ylabel("Score", fontsize=12, fontweight="bold")
ax.set_title("Performance Comparison: DoRA vs LoRA\nBanking77 Intent Classification", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n").title() for m in metrics_names], fontsize=10)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3, linestyle="--")

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig("outputs/plots/performance_comparison.png", dpi=300, bbox_inches="tight")
print("\n✅ Comparison plot saved to: outputs/plots/performance_comparison.png")
plt.show()

# 8. WINNER SUMMARY
print("\n" + "="*100)
print("🎖️  OVERALL WINNER SUMMARY")
print("="*100 + "\n")

dora_wins = sum(1 for m in comparison_df.index if dora_metrics[m] > lora_metrics[m])
lora_wins = len(comparison_df.index) - dora_wins

print(f"📈 DoRA Model Wins: {dora_wins}/{len(comparison_df.index)} metrics")
print(f"📉 LoRA Model Wins: {lora_wins}/{len(comparison_df.index)} metrics")

if dora_wins > lora_wins:
    avg_improvement = (comparison_df["Improvement (%)"].mean())
    print(f"\n🏆 DoRA is SUPERIOR on average by {avg_improvement:.2f}%")
else:
    print(f"\n⚠️  LoRA performs better on {lora_wins} metrics")

print("\n" + "="*100)


🏆 PERFORMANCE COMPARISON (LEADERBOARD) 🏆



,LoRA Model (rajo0113),DoRA Model (tpdinh2612),Delta,Improvement (%)
exact_match_accuracy,0.8744,0.9231,+0.0487,+5.57%
macro_f1,0.7493,0.8377,+0.0883,+11.79%
macro_precision,0.7567,0.8420,+0.0853,+11.27%
macro_recall,0.7481,0.8362,+0.0881,+11.78%


## 7. Per-Class Improvement Analysis

In [ ]:
# Load per-class classification reports
lora_report_df = pd.read_csv("outputs/metrics/LoRA_Model_per_class_report.csv", index_col=0)
dora_report_df = pd.read_csv("outputs/metrics/DoRA_Model_per_class_report.csv", index_col=0)

# Extract per-class F1 scores (excluding summary rows)
lora_per_class = lora_report_df[:-3]["f1-score"]  # Exclude accuracy, macro avg, weighted avg
dora_per_class = dora_report_df[:-3]["f1-score"]

# Compute improvements
per_class_improvement = dora_per_class - lora_per_class
per_class_rel_improvement = (per_class_improvement / (lora_per_class + 1e-10)) * 100

# Create improvement dataframe
improvement_df = pd.DataFrame({
    "Intent": per_class_improvement.index,
    "LoRA F1": lora_per_class.values.round(4),
    "DoRA F1": dora_per_class.values.round(4),
    "Δ F1": per_class_improvement.values.round(4),
    "Improvement (%)": per_class_rel_improvement.values.round(2),
    "Status": ["🔼 DoRA" if x > 0 else "🔽 LoRA" for x in per_class_improvement.values]
})

# Sort by improvement descending
improvement_df = improvement_df.sort_values("Δ F1", ascending=False)

print("\n" + "="*120)
print("📊 TOP 15 CLASSES WHERE DoRA EXCELS")
print("="*120 + "\n")

try:
    display(improvement_df.head(15))
except NameError:
    print(improvement_df.head(15).to_markdown(index=False))

print("\n" + "="*120)
print("📊 TOP 10 CLASSES WHERE LoRA EXCELS")
print("="*120 + "\n")

try:
    display(improvement_df.tail(10))
except NameError:
    print(improvement_df.tail(10).to_markdown(index=False))

# Visualization: Top 20 improvements
fig, ax = plt.subplots(figsize=(14, 8))

top_improvements = improvement_df.head(20)
colors = ["#2ECC71" if x > 0 else "#E74C3C" for x in top_improvements["Δ F1"]]

bars = ax.barh(range(len(top_improvements)), top_improvements["Δ F1"], color=colors, alpha=0.8)
ax.set_yticks(range(len(top_improvements)))
ax.set_yticklabels(top_improvements["Intent"], fontsize=10)
ax.set_xlabel("F1 Score Improvement (DoRA - LoRA)", fontsize=12, fontweight="bold")
ax.set_title("Top 20 Intent Classes: DoRA Improvement over LoRA", fontsize=14, fontweight="bold")
ax.axvline(x=0, color="black", linestyle="-", linewidth=0.8)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Add value labels
for i, (bar, val) in enumerate(zip(bars, top_improvements["Δ F1"])):
    ax.text(val, i, f" {val:.4f}", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig("outputs/plots/per_class_improvement_top20.png", dpi=300, bbox_inches="tight")
print("\n✅ Per-class improvement plot saved to: outputs/plots/per_class_improvement_top20.png")
plt.show()

# Statistics
dora_better = sum(per_class_improvement > 0)
lora_better = sum(per_class_improvement <= 0)
print(f"\n📈 Class-Level Statistics:")
print(f"   • DoRA wins on {dora_better}/{len(per_class_improvement)} classes")
print(f"   • LoRA wins on {lora_better}/{len(per_class_improvement)} classes")
print(f"   • Average improvement: {per_class_rel_improvement.mean():.2f}%")
print(f"   • Max improvement: {per_class_rel_improvement.max():.2f}% ({improvement_df.iloc[0]['Intent']})")
print(f"   • Min improvement: {per_class_rel_improvement.min():.2f}% ({improvement_df.iloc[-1]['Intent']})")